In [8]:
from openai import OpenAI
import os
token = ''
client = OpenAI(api_key=token)

import pymupdf
import os


def summarization_gen(instruction, model_temp, model_name):
    response = client.chat.completions.create(
        model=model_name,  # Or the specific model you have access to
        messages=[
            {"role": "system", "content": "You are a scientific assistant specializing in molecular biology and bioinformatics. Your goal is to generate concise, structured gene descriptions based on extracted scientific data."},
            {"role": "user", "content": f"{instruction}"}
        ],
    )
    
    # Extract the response
    # print(response)
    information = response.choices[0].message.content.strip()
    
    return information




def create_prompt_from_text(existing_description, extracted_info_json):
    instruction = f"""
    ## **Task Description**
    Update the existing gene description using the extracted information below. Maintain the structure and clarity of the original description while integrating newly extracted informaiton. Expand on incomplete sections and refine content as needed.

    ## **Existing Gene Description**:
    {existing_description}

    ## **Extracted Information**:
    {extracted_info_json}

    ## **Guidelines for Updating the Description**
    - **Preserve and refine**: Keep relevant information from the existing description and improve readability where necessary.
    - **Incorporate newly extracted information**: Use the extracted information to enhance the summary, ensuring all relevant sections are updated.
    - **Summarize effectively**: Group multiple terms into **high-level categories** to avoid excessive listing.
    - **Follow structured formatting**: Adhere to the sentence templates below to maintain consistency and modularity.

    ## **Sentence Templates for Gene Description**:
    - **Molecular Function**: Exhibits <list of gene ontology molecular function terms>.
    - **Biological Process**: Involved in **broad biological categories** such as <high-level biological processes>.
    - **Subcellular Localization**: Localizes to <list of gene ontology cellular component terms>.
    - **Pathways**: Functions in <list of pathway terms>.
    - **Disease Relevance**: Used to study <list of disease ontology terms>.
    - **Expression**: Expressed in <list of anatomy ontology terms>.
    - **Implication of Human Orthologs**: Human ortholog(s) of this gene implicated in <list of disease ontology terms>.
    - **Human Orthologs**: Orthologous to <list of human orthologs>.
    - **Modifiers**: <Modifier> ameliorates or exacerbates the disease phenotype or condition.

    """

    return instruction

In [9]:
import os
import json
from openai import OpenAI
from pprint import pprint

with open('data_pairs_20250510.json', 'r') as f:
    gene_data = json.load(f)

folder = '/Users/pengyuan/Desktop/GeneSum/alliance_files/gene_descriptions/incomplete_desc'
models = ['gpt-4-turbo', 'o1', 'gpt-4.5-preview', 'gpt-4o', 'gpt-4o-mini']

for gene_name in gene_data.keys():
    task_name = gene_name+ '_InCompleteDesc_paperinfo_20250505.json'
    if os.path.join(folder, task_name):
        print(gene_name)
        paper_info = {}
        paper_info['gene_name'] = gene_name
        for key in gene_data[gene_name]['paper_inf'].keys():
            if len(gene_data[gene_name]['paper_inf'][key])>0:
                print(gene_data[gene_name]['paper_inf'][key])
                for item in gene_data[gene_name]['paper_inf'][key]:
                    paper_info[key] = [item['sentence template'] for item in gene_data[gene_name]['paper_inf'][key] if 'sentence template' in item.keys()]
        instruction = create_prompt_from_text(gene_data[gene_name]['gt']['description'], paper_info)
        # print(instruction) 

        model_temp = 0.0
        
        gene_info = {}
        for model_name in models:
            gene_info[model_name] = summarization_gen(instruction, model_temp, model_name)
        # print('*'*10)
        # print(gene_info)
        # print('*'*10)
        task_name = gene_name+ '_InCompleteDesc_paperinfo_20250505.json'
        with open(os.path.join(folder, task_name), 'w') as f:
            json.dump(gene_info, f, indent=4)

daf-16
[{'sentence template': 'DAF-16 is involved in the determination of adult lifespan (GO:0008340).', 'evidence_sentences': 'The insulin signaling pathway is an important lifespan regulation pathway, DAF-16 is an important … DAF-16 is expressed in large amounts, which can delay aging and prolong the lifespan of worms…”', 'references': 'PMID_38841104'}, {'sentence template': 'DAF-16 is involved in the response to oxidative stress (GO:0006979).', 'evidence_sentences': '…which contributed significantly to inhibition of Aβ toxicity and enhancement of antioxidative stress. … can be used to inhibit AD development by catalyzing the removal of O2− to protect the organism from oxidative stress…”', 'references': 'PMID_38841104'}, {'sentence template': 'DAF-16 is involved in the negative regulation of amyloid-beta toxicity.', 'evidence_sentences': 'Finally, we also found that p-HBA promoted nuclear localization of DAF-16 and increased the expression of SKN-1, SOD-3, and GST-4, which contribute